### Library Imports

This section imports the **libraries and utilities** required throughout the training notebook.

PyTorch provides the neural-network, optimization, tensor, and gradient-management functionality, while the `tokenizers` library is used to process text into token IDs. `numpy` is used to reconstruct the stored binary token sequences, and `mysql.connector` provides access to the database containing the training data.

Additional modules provide functionality for **random sampling, filesystem management, environment configuration, mathematical operations, and system-level utilities**. The gradient clipping function is also aliased to `clip_gradients` to make its purpose clearer when it is used in the training loop.

In [74]:
from torch.nn.utils import clip_grad_norm_ as clip_gradients
import torch
from torch import nn
from torch.optim import Optimizer

import random
import sys
import os

from math import exp
from pathlib import Path

from dotenv import load_dotenv

from tokenizers import Tokenizer

import mysql.connector
import numpy


### Environment Configuration

This function loads the **environment variables** defined in the project's `.env` file into the application's environment.

These variables can be used to store **configuration values and sensitive information**, such as database credentials, without placing them directly inside the source code.

In [ ]:
load_dotenv()

### Library Imports

This section imports the **libraries and utilities** required throughout the training notebook.

PyTorch provides the neural-network, optimization, tensor, and gradient-management functionality, while the `tokenizers` library is used to process text into token IDs. `numpy` is used to reconstruct the stored binary token sequences, and `mysql.connector` provides access to the database containing the training data.

Additional modules provide functionality for **random sampling, filesystem management, environment configuration, mathematical operations, and system-level utilities**. The gradient clipping function is also aliased to `clip_gradients` to make its purpose clearer when it is used in the training loop.

In [76]:
def load_english_wiki_batch(cursor, table: str , batch_size: int) -> tuple:

    cursor.execute(f"SELECT MIN(id), MAX(id) FROM {table}")
    min_id, max_id = cursor.fetchone()

    if not min_id or not max_id:
        raise ValueError("The Table Is Empty")

    ids = [random.randint(min_id, max_id) for _ in range(batch_size)]
    placeholders = ','.join(['%s'] * batch_size)

    cursor.execute(f"SELECT sample FROM {table} WHERE id IN ({placeholders});", ids)

    samples = [row[0] for row in cursor.fetchall() if row[0] is not None]

    while len(samples) < batch_size:
        random_id = random.randint(min_id, max_id)

        cursor.execute(f"SELECT sample FROM {table} WHERE id = %s;", (random_id,))
        sample = cursor.fetchone()

        if sample is not None:
            samples.append(sample[0])

    batch_input, batch_output = [], []
    for sample in samples:
        tokens = numpy.frombuffer(sample, dtype=numpy.uint16).tolist()

        batch_input.append(tokens[:512])
        batch_output.append(tokens[1:])

    batch_input = torch.tensor(batch_input, dtype=torch.long)
    batch_output = torch.tensor(batch_output, dtype=torch.long)

    return batch_input, batch_output

### Feed-Forward Network

This class implements the **position-wise feed-forward network** used within each Transformer block.

The network first expands each token representation from the model dimension, `d_model`, to a larger hidden dimension, `d_hidden`. It then applies the **GELU activation function** to introduce non-linearity before projecting the representation back to `d_model`.

Two `Dropout` layers are used to help **reduce overfitting** during training. Since this network operates independently on each token position, it allows the Transformer to perform additional **non-linear transformations** on the representations produced by the attention mechanism.

In [77]:
class MultiLayerPerceptron(nn.Module):

    def __init__(self, d_model: int, d_hidden: int) -> None:

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(d_model, d_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_hidden, d_model),
            nn.Dropout(0.1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


### Transformer Block

This class implements a **single Transformer block**, combining **multi-head self-attention** with a **position-wise feed-forward network**.

The block uses **Pre-Layer Normalization**, where `LayerNorm` is applied before each major sub-layer. The input is first normalized and passed through `MultiheadAttention`, allowing each token to incorporate information from other positions in the sequence.

The attention output is then combined with the original input through a **residual connection**. This is followed by a second `LayerNorm` and the **multi-layer perceptron (MLP)**, which performs additional non-linear transformations on each token representation.

Finally, another **residual connection** adds the MLP output back to the representation produced by the attention sub-layer. These residual connections help **preserve information and improve gradient flow**, making it possible to stack multiple Transformer blocks to form the complete model.

In [78]:
class TransformerBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int) -> None:

        super().__init__()

        self.first_layer_norm = nn.LayerNorm(d_model)
        self.multi_head_attention = nn.MultiheadAttention(d_model, num_heads, dropout=0.1, batch_first=True)

        self.second_layer_norm = nn.LayerNorm(d_model)
        self.multi_layer_perceptron = MultiLayerPerceptron(d_model, 4 * d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor|None =None) -> torch.Tensor:

        output = self.first_layer_norm(x)
        output = self.multi_head_attention(query=output, key=output, value=output, attn_mask=mask)[0]

        residual = x + output

        output = self.second_layer_norm(residual)
        output = self.multi_layer_perceptron(output)

        output = output + residual

        return output


### Transformer Model

This class defines the **complete GPT-style Transformer**, combining token embeddings, positional embeddings, multiple Transformer blocks, and a final output projection that produces a probability distribution over the vocabulary for each position in the sequence.

The input token IDs are first converted into **token embeddings** and combined with **learned positional embeddings**. This provides the model with both the semantic representation of each token and information about its position within the sequence. A **causal attention mask** is also registered as a buffer, preventing each token from attending to future tokens during autoregressive training.

The resulting representations are passed sequentially through the collection of **Transformer blocks**. After the final block, a `LayerNorm` is applied before the representations are projected back into the vocabulary space by the `output_layer`. The resulting logits have the shape `(batch_size, sequence_length, vocab_size)` and represent the model's predictions for the **next token at every position**.

The output projection shares its weights with the token embedding layer through **weight tying**. This allows the same learned representation to be used for both converting token IDs into embeddings and converting hidden representations back into vocabulary predictions, while also reducing the number of trainable parameters.

Finally, `initialize_weights` provides the model's **weight initialization strategy**. Linear and embedding weights are initialized from a normal distribution with a standard deviation of `0.02`, linear biases are initialized to zero, and LayerNorm parameters are initialized to their standard values. The method is applied recursively to the model using `self.apply()`.

In [79]:
class Transformer(nn.Module):

    def __init__(self, d_model: int, vocab_size: int, num_heads: int, sequence_length: int, num_transformer_blocks: int) -> None:

        super().__init__()

        self.token_embeddings = nn.Embedding(vocab_size, d_model)
        self.position_embeddings = nn.Embedding(sequence_length, d_model)

        self.dropout = nn.Dropout(0.1)

        mask = torch.triu(torch.full((sequence_length, sequence_length), float('-inf')), diagonal=1)
        self.register_buffer("mask", mask)

        self.transformer_blocks = nn.ModuleList([TransformerBlock(d_model, num_heads) for _ in range(num_transformer_blocks)])

        self.layer_norm = nn.LayerNorm(d_model)
        self.output_layer = nn.Linear(d_model, vocab_size)

        self.apply(self.initialize_weights)

        self.output_layer.weight = self.token_embeddings.weight

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        token_embeddings = self.token_embeddings(x)
        position_embeddings = self.position_embeddings(torch.arange(x.shape[1], device=x.device)).unsqueeze(0)
        transformer_input = token_embeddings + position_embeddings

        transformer_output = self.dropout(transformer_input)

        mask = self.mask[:x.shape[1], :x.shape[1]]
        for transformer_block in self.transformer_blocks:
            transformer_output = transformer_block(transformer_output, mask=mask)

        transformer_output = self.layer_norm(transformer_output)
        transformer_output = self.output_layer(transformer_output)

        return transformer_output

    @staticmethod
    def initialize_weights(module):

        with torch.no_grad():
            if isinstance(module, (nn.Linear, nn.Embedding)):
                module.weight.normal_(mean=0.0, std=0.02)

                if isinstance(module, nn.Linear) and module.bias is not None:
                    module.bias.zero_()

            elif isinstance(module, nn.LayerNorm):
                module.bias.zero_()
                module.weight.fill_(1.0)

### Warmup-Stable-Decay Learning Rate Scheduler

This class implements a custom **Warmup-Stable-Decay (WSD) learning rate scheduler** designed to control the learning rate throughout training. The scheduler divides the training process into three possible modes: **warmup (`W`)**, **stable (`S`)**, and **decay (`D`)**.

During **warmup**, the learning rate is gradually increased from `minimum_ratio × learning_rate` toward the configured maximum learning rate. During the **stable** phase, the learning rate remains unchanged, while during **decay**, it is gradually reduced toward the minimum learning rate.

The learning rate change per step is determined by `delta`, which distributes the difference between the minimum and maximum learning rates evenly across the specified number of `transition_steps`. The scheduler directly updates the learning rate of every parameter group in the optimizer.

The scheduler also provides **state management** through `state_dict()` and `load_state_dict()`. This allows the current learning rate, number of scheduler steps, and current mode to be saved and restored, which is particularly useful when training the model across **multiple independent training sessions**.

The `set_mode()` method additionally validates the requested phase, ensuring that only the supported modes—`W`, `S`, and `D`—can be selected.

In [80]:
class WarmupStableDecayLRScheduler:

    def __init__(self, optimizer : Optimizer, mode: str, transition_steps: int, learning_rate: float | int, minimum_ratio: float | int = 0.05) -> None:

        if transition_steps <= 0 or learning_rate <= 0 or not 0 < minimum_ratio <= 1:
            sys.exit(f"Invalid Arguments, Please Check The Argument Report For More Details\n\n--------------- Arguments Report ---------------\n\nTransitions Steps Greater Than 0: {transition_steps > 0}\nLearning Rate Greater Than 0: {learning_rate > 0}\nMinimum Ratio Between 0 And 1: {1 >= minimum_ratio >= 0}")

        self.optimizer = optimizer
        self.mode = mode.upper()

        self.min_learning_rate = learning_rate * minimum_ratio
        self.max_learning_rate = learning_rate

        self.learning_rate = self.min_learning_rate

        self.delta = (self.max_learning_rate - self.min_learning_rate) / transition_steps

        self.steps = 0

    def step(self) -> None:

        if self.mode == 'W':
            updated_learning_rate = min(self.learning_rate + self.delta, self.max_learning_rate)
        elif self.mode == 'D':
            updated_learning_rate = max(self.learning_rate - self.delta, self.min_learning_rate)
        else:
            updated_learning_rate = self.learning_rate

        self.set_learning_rate(updated_learning_rate)

        self.steps += 1

    def set_learning_rate(self, learning_rate: float|int) -> None:

        self.learning_rate = learning_rate

        for param_group in self.optimizer.param_groups:
            param_group['lr'] = learning_rate

    def set_mode(self, mode: str) -> None:

        mode = mode.strip().upper()

        if mode not in ('W', 'S', 'D'):
            raise ValueError('Unknown scheduler mode. Please use one of W (Warm Up), S (Stable), or D (Decay).')

        self.mode = mode

    def state_dict(self) -> dict:

        return {
            "learning_rate": self.learning_rate,
            "steps": self.steps,
            "mode": self.mode,
        }

    def load_state_dict(self, state_dict:dict) -> None:

        self.steps = state_dict["steps"]

        self.set_learning_rate(state_dict["learning_rate"])
        self.set_mode(state_dict["mode"])


### Training Configuration and Model Initialization

This section defines the **hyperparameters and configuration** used throughout training. The learning rate, weight decay, model dimensions, sequence length, batch size, learning-rate transition length, and gradient clipping threshold are centralized here so they can be easily modified without changing the training logic.

The model uses a `d_model` of `512`, **8 Transformer blocks**, and **8 attention heads**, with a maximum sequence length of `512` tokens. Training uses batches of `8` sequences, while `AdamW` provides parameter optimization with a weight decay of `0.1`.

The **GPT-2 tokenizer** is loaded using the Hugging Face `tokenizers` library, and its vocabulary size is used to determine the size of the model's embedding and output layers. The model is then moved to either a **CUDA GPU or CPU**, depending on hardware availability.

A custom **Warmup-Stable-Decay learning-rate scheduler** is initialized in warmup mode, beginning at `5%` of the configured learning rate and gradually increasing toward `3e-4`. `CrossEntropyLoss` is used as the training objective for **next-token prediction**.

Finally, a metrics dictionary is created to store important training information, including **tokens processed, training and validation loss, and perplexity**. The checkpoint path is also defined so the model and training state can be saved and restored across multiple training sessions.

In [81]:
# ------- Configuration ------- #
LEARNING_RATE = 3e-4
MINIMUM_LEARNING_RATE_RATIO = 0.05

WEIGHT_DECAY = 0.1

D_MODEL = 512
NUMBER_OF_TRANSFORMER_BLOCKS = 8
NUMBER_OF_HEADS = 8

SEQUENCE_LENGTH = 512
BATCH_SIZE = 8

MODE_TRANSITION_STEPS = 10000

GRAD_CLIP_VALUE = 1.0
# ------------------------------#

# --------------- Tokenizer --------------- #
tokenizer = Tokenizer.from_pretrained('gpt2')
vocab = tokenizer.get_vocab_size()
# ----------------------------------------- #

# --------------------- Model Initialization --------------------- #
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Transformer(
    d_model=D_MODEL,
    vocab_size=vocab,
    num_transformer_blocks=NUMBER_OF_TRANSFORMER_BLOCKS,
    sequence_length=SEQUENCE_LENGTH,
    num_heads=NUMBER_OF_HEADS).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = WarmupStableDecayLRScheduler(
    optimizer,
    mode='W',
    transition_steps=MODE_TRANSITION_STEPS,
    learning_rate=LEARNING_RATE,
    minimum_ratio=MINIMUM_LEARNING_RATE_RATIO,
)

criterion = nn.CrossEntropyLoss()
# ---------------------------------------------------------------- #

# --------- Metrics --------- #
metrics = {
    'tokens_seen': [],
    'training_loss': [],
    'validation_loss': [],
    'training_perplexity': [],
    'validation_perplexity': [],
}
# --------------------------- #

# ------------- Path ------------- #
path = Path("Pisces.pth")
# -------------------------------- #

### Validation

This function evaluates the model's performance on a separate **validation dataset** without updating its parameters. The model is first switched to evaluation mode using `model.eval()`, which ensures that layers such as **Dropout** behave deterministically during evaluation.

For each validation batch, the function loads a set of tokenized sequences, moves the input and target tensors to the selected device, and performs a forward pass through the model. The resulting logits are transposed so that they match the input format expected by `CrossEntropyLoss`, which requires the class dimension to come before the sequence dimension.

The loss from each batch is accumulated and then averaged across all validation batches to produce the final **validation loss**. Gradient calculation is disabled with `torch.no_grad()` because no parameter updates are performed during validation, reducing unnecessary memory usage and computation.

After validation is complete, the model is returned to **training mode** with `model.train()` so that subsequent training steps correctly re-enable training-specific behavior such as Dropout.

In [82]:
def validate(cursor, samples: int, validation_table: str):

    model.eval()

    validation_loss = 0

    with torch.no_grad():
        for counter in range(samples):
            x, y = load_english_wiki_batch(cursor, validation_table, BATCH_SIZE)
            x, y = x.to(device), y.to(device)

            logits = model(x).transpose(1, 2)

            loss = criterion(logits, y)

            validation_loss = validation_loss + loss.item()

    validation_loss = validation_loss / samples

    model.train()

    return validation_loss

### Checkpoint Saving

This function creates and saves a **training checkpoint** containing the complete state required to resume training from the current point.

The checkpoint stores the model parameters through `model_state_dict`, along with the internal state of the **AdamW optimizer** and the custom learning-rate scheduler. Saving the optimizer and scheduler states is important because they contain information that affects how training continues, rather than just the model's learned weights.

The function also saves the accumulated **training metrics**, including the number of tokens processed, training and validation loss, and their corresponding perplexity values. This allows the training history to be preserved across separate training sessions.

Finally, `torch.save()` serializes the checkpoint and writes it to the path defined by `path`, allowing the complete training state to be restored later.

In [83]:
def save():

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),

        'tokens_seen': metrics['tokens_seen'],

        'training_loss': metrics['training_loss'],
        'validation_loss': metrics['validation_loss'],

        'training_perplexity': metrics['training_perplexity'],
        'validation_perplexity': metrics['validation_perplexity'],
    }
    torch.save(checkpoint, path)

### Checkpoint Loading

This function restores the previously saved **training checkpoint** if one exists at the configured path.

The checkpoint is loaded using `torch.load()` with `map_location=device`, allowing the saved tensors to be loaded onto the **currently available device**, regardless of which device was used when the checkpoint was created.

The saved **model, optimizer, and learning-rate scheduler states** are then restored, ensuring that training can continue from approximately the same state rather than starting over. The accumulated training metrics are also restored so that the training history remains continuous across multiple training sessions.

If no checkpoint is found, the function simply reports that training will begin using the **default initialized state**.

In [84]:
def load():

    if path.exists():
        data = torch.load(path, map_location=device)

        model.load_state_dict(data['model_state_dict'])
        optimizer.load_state_dict(data['optimizer_state_dict'])
        scheduler.load_state_dict(data['scheduler_state_dict'])

        metrics["tokens_seen"] = data['tokens_seen']

        metrics["training_loss"] = data['training_loss']
        metrics["validation_loss"] = data['validation_loss']

        metrics["training_perplexity"] = data['training_perplexity']
        metrics["validation_perplexity"] = data['validation_perplexity']
    else:
        print(f"No checkpoint found at {path}, Initializing Default Values")


### Training Loop

This function manages a complete **training session**, including database access, checkpoint restoration, model training, validation, metric calculation, and checkpoint saving.

The function first establishes a connection to the **MySQL database** and restores any previously saved checkpoint. This allows training to continue across multiple sessions while preserving the model parameters, optimizer state, scheduler state, and previously recorded metrics.

During each training iteration, a batch of tokenized sequences is loaded and transferred to the selected device. The model produces **next-token logits**, which are compared against the target sequences using `CrossEntropyLoss`. The resulting loss is backpropagated, the gradients are constrained using **gradient clipping**, and the optimizer updates the model parameters. The learning-rate scheduler is then advanced by one step.

After the training iterations are complete, the average **training loss** and corresponding **perplexity** are calculated. The model is then evaluated on the validation dataset to measure how well it generalizes to unseen samples. These results, along with the number of tokens processed, are added to the accumulated training metrics.

A training report is printed at the end of the session, providing a concise overview of the model's current performance. Finally, the complete state is saved to a **checkpoint**, allowing the next training session to resume from where this one ended. The database cursor and connection are also closed in the `finally` block to ensure that resources are released even if an error occurs.

In [85]:
def train(database: dict, training_samples: int, validation_samples: int, training_table: str, validation_table: str):

    connection, cursor = None, None

    try:
        connection = mysql.connector.connect(**database)
        cursor = connection.cursor()

        load()

        model.train()

        training_loss = 0

        for counter in range(training_samples):
            x, y = load_english_wiki_batch(cursor, training_table, BATCH_SIZE)
            x, y = x.to(device), y.to(device)

            logits = model(x).transpose(1, 2)

            loss = criterion(logits, y)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()

            clip_gradients(model.parameters(), GRAD_CLIP_VALUE)

            optimizer.step()
            scheduler.step()

            training_loss = training_loss + loss.item()

        training_loss = training_loss / training_samples
        training_perplexity = exp(training_loss)

        validation_loss = validate(cursor, validation_samples, validation_table)
        validation_perplexity = exp(validation_loss)

        tokens_seen = BATCH_SIZE * SEQUENCE_LENGTH * training_samples

        metrics['tokens_seen'].append(tokens_seen)
        metrics['training_loss'].append(training_loss)
        metrics['validation_loss'].append(validation_loss)
        metrics['training_perplexity'].append(training_perplexity)
        metrics['validation_perplexity'].append(validation_perplexity)

        print(f"------------------------------------------- REPORT --------------------------------------------")
        print(f"| Training Loss | Training Perplexity | Validation Loss | Validation Perplexity | Tokens Seen |")
        print(f"| ------------- | ------------------- | --------------- | --------------------- | ----------- |")
        print(f"| {training_loss:^13.5f} | {training_perplexity:^19.5f} | {validation_loss:^15.5f} | {validation_perplexity:^21.5f} | {tokens_seen:^11} |")
        print(f"-----------------------------------------------------------------------------------------------")

        save()

    except mysql.connector.Error as error:
        sys.exit(f"Error while connecting to MySQL: {error}")

    finally:
        if cursor is not None:
            cursor.close()
        if connection is not None:
            connection.close()

### Main Block

In [105]:
train(
    database={
    'host': '5.tcp.eu.ngrok.io',
    'password': os.getenv('DB_PASSWORD'),
    'port': 19925,
    'user': os.getenv('DB_USER'),
    'database': os.getenv('DB_NAME')
    },
    training_samples=100,
    validation_samples=10,
    training_table="EnglishWikiTrainSamples",
    validation_table="EnglishWikiTestSamples",
    )

------------------------------------------- REPORT --------------------------------------------
| Training Loss | Training Perplexity | Validation Loss | Validation Perplexity | Tokens Seen |
| ------------- | ------------------- | --------------- | --------------------- | ----------- |
|    6.97694    |     1071.63323      |     7.03384     |      1134.37653       |   409600    |
-----------------------------------------------------------------------------------------------
